# 🫁 Medical Clinical RAG — NICE Asthma Hackathon Final

**Scope:** Asthma within allergic diseases — adults, young people and children, with age-specific recommendations preserved.

**Guideline:** *Asthma: diagnosis, monitoring and chronic asthma management (BTS, NICE, SIGN)* — NICE NG245.

This notebook keeps the original MedCPT + FAISS + BM25 + MMR approach and adds the missing Hackathon requirements:

- PDF/page-aware ingestion
- Structure/recommendation-aware chunks
- Metadata and citations
- Hybrid retrieval
- Top-K + chunk-size experimentation
- Relevance threshold / abstention
- Out-of-scope protection
- 20-question labeled retrieval evaluation
- Precision@3 and Precision@5
- Evidence panel
- Citation + numerical validation
- LLM generation gate
- Failure-case tests
- Saving RAG artifacts

> **Important:** This is a prototype for the Hackathon. It is not a diagnostic or prescribing tool.

In [1]:
# 1) Install dependencies — Google Colab
!pip -q install pymupdf faiss-cpu rank-bm25 sentence-transformers transformers accelerate openai pandas numpy scikit-learn


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\Mega Store\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [2]:
# 2) Locate the NICE PDF
from pathlib import Path
import os

PDF_CANDIDATES = [
    "asthma-diagnosis-monitoring-and-chronic-asthma-management-bts-nice-sign-pdf-66143958279109.pdf",
    "/content/asthma-diagnosis-monitoring-and-chronic-asthma-management-bts-nice-sign-pdf-66143958279109.pdf",
]

PDF_PATH = next((p for p in PDF_CANDIDATES if Path(p).exists()), None)

if PDF_PATH is None:
    print("NICE PDF not found automatically.")
    print("Upload the NICE NG245 PDF to Colab, then set PDF_PATH to its filename.")
else:
    print("Using:", PDF_PATH)

Using: asthma-diagnosis-monitoring-and-chronic-asthma-management-bts-nice-sign-pdf-66143958279109.pdf


## 3. Imports and configuration

In [1]:
import re
import json
import math
import pickle
import numpy as np
import pandas as pd
import fitz
import torch
import faiss

from rank_bm25 import BM25Okapi
from pathlib import Path
from IPython.display import display, Markdown

DOCUMENT_ID = "NICE_NG245"
DOCUMENT_TITLE = "Asthma: diagnosis, monitoring and chronic asthma management (BTS, NICE, SIGN)"
SOURCE_AUTHORITY = "NICE / BTS / SIGN"
POPULATION = "Adults, young people and children (age-specific recommendations apply)"
CLINICAL_FIELD = "Allergic Diseases"
CONDITION = "Asthma"
SOURCE_URL = "https://www.nice.org.uk/guidance/ng245"

# Default chunk configuration
MAX_CHUNK_TOKENS = 450
CHUNK_OVERLAP_TOKENS = 0

# Retrieval
DENSE_TOP_K = 30
BM25_TOP_K = 30
FINAL_TOP_K = 5
MMR_LAMBDA = 0.70

# IMPORTANT: calibrated later with the evaluation set.
MIN_RELEVANCE_SCORE = 0.35

# Hybrid weighting
DENSE_WEIGHT = 0.65
BM25_WEIGHT = 0.35

# Optional LLM
USE_LLM = False
OPENAI_MODEL = "gpt-4.1-mini"

print(f"Scope: {CLINICAL_FIELD} → {CONDITION}")
print(f"Guideline: {DOCUMENT_ID}")
print(f"Retrieval threshold: {MIN_RELEVANCE_SCORE}")

Scope: Allergic Diseases → Asthma
Guideline: NICE_NG245
Retrieval threshold: 0.35


## 4. PDF ingestion — preserve pages and clinical wording

In [3]:
from pathlib import Path

PDF_PATH = Path(
    "asthma-diagnosis-monitoring-and-chronic-asthma-management-bts-nice-sign-pdf-66143958279109.pdf"
)

In [4]:
print("PDF exists:", PDF_PATH.exists())
print("PDF path:", PDF_PATH.resolve())

PDF exists: True
PDF path: D:\Download\asthma-diagnosis-monitoring-and-chronic-asthma-management-bts-nice-sign-pdf-66143958279109.pdf


In [5]:
def normalize_clinical_text(text: str) -> str:
    text = text.replace("\u00ad", "")
    text = re.sub(r"-\n", "", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

if PDF_PATH is None:
    raise FileNotFoundError(
        "Please upload the NICE NG245 PDF to Colab and set PDF_PATH before running this cell."
    )

raw_pages = []
pdf = fitz.open(PDF_PATH)

for page_num, page in enumerate(pdf, start=1):
    text = normalize_clinical_text(page.get_text("text"))
    if text:
        raw_pages.append({"page": page_num, "text": text})

pdf.close()

print(f"Pages extracted: {len(raw_pages)}")
print(raw_pages[0]["text"][:1200])

Pages extracted: 64
Asthma: diagnosis, 
monitoring and chronic 
asthma management (BTS, 
NICE, SIGN) 
NICE guideline 
Published: 27 November 2024 
www.nice.org.uk/guidance/ng245 
© NICE 2026. All rights reserved. Subject to Notice of rights (https://www.nice.org.uk/terms-andconditions#notice-of-rights).


## 5. Structure-aware clinical chunking

In [6]:
SECTION_RE = re.compile(
    r"(?m)^\s*(1\.\d+(?:\.\d+)?)\s+(.+?)\s*$"
)

RECOMMENDATION_RE = re.compile(
    r"(?m)(?<!\d)(\d+\.\d+\.\d+)\s+"
)

def token_count(text):
    return max(1, len(re.findall(r"\S+", text)))

def split_by_sentences(text, max_tokens=MAX_CHUNK_TOKENS, overlap_tokens=CHUNK_OVERLAP_TOKENS):
    sentences = re.split(r"(?<=[.!?])\s+", text.strip())
    chunks = []
    current = []

    for sentence in sentences:
        if not sentence:
            continue

        candidate = " ".join(current + [sentence])

        if current and token_count(candidate) > max_tokens:
            chunks.append(" ".join(current).strip())

            if overlap_tokens > 0:
                words = " ".join(current).split()
                current = words[-overlap_tokens:] + [sentence]
                current = [" ".join(current)]
            else:
                current = [sentence]
        else:
            current.append(sentence)

    if current:
        chunks.append(" ".join(current).strip())

    return [c for c in chunks if c]

def current_section_before(text, position):
    matches = list(SECTION_RE.finditer(text[:position]))
    if not matches:
        return {"section_id": "general", "section_title": "General guideline context"}

    m = matches[-1]
    return {
        "section_id": m.group(1),
        "section_title": m.group(2).strip()
    }

def infer_population(section_title, text):
    s = (section_title + " " + text).lower()

    if "children under 5" in s:
        return "Children under 5"
    if "children aged 5 to 11" in s:
        return "Children aged 5 to 11"
    if "children aged 5 to 16" in s:
        return "Children aged 5 to 16"
    if "aged 12 and over" in s:
        return "People aged 12 and over"
    if "adolescents" in s:
        return "Adolescents"
    if "pregnancy" in s or "breastfeeding" in s:
        return "People with asthma during pregnancy/breastfeeding"
    if "adults" in s:
        return "Adults"

    return POPULATION

def add_chunk(chunks, counter, text, page, section, recommendation_id, population):
    text = text.strip()
    if not text:
        return counter

    counter += 1

    context_prefix = (
        f"Condition: {CONDITION}. "
        f"Section {section['section_id']}: {section['section_title']}. "
        f"Population: {population}. "
        f"Recommendation: {recommendation_id}. "
    )

    chunks.append({
        "chunk_id": f"{DOCUMENT_ID}_p{page}_{counter:04d}",
        "text": context_prefix + text,
        "source_text": text,
        "page": page,
        "section_id": section["section_id"],
        "section": section["section_title"],
        "recommendation_id": recommendation_id,
        "document_id": DOCUMENT_ID,
        "document_type": "clinical_guideline",
        "clinical_field": CLINICAL_FIELD,
        "condition": CONDITION,
        "source_authority": SOURCE_AUTHORITY,
        "population": population,
        "title": DOCUMENT_TITLE,
        "source_url": SOURCE_URL
    })

    return counter

def make_chunks(pages, max_tokens=MAX_CHUNK_TOKENS, overlap_tokens=CHUNK_OVERLAP_TOKENS):
    chunks = []
    chunk_counter = 0

    for page in pages:
        text = page["text"]
        matches = list(RECOMMENDATION_RE.finditer(text))

        if not matches:
            section = current_section_before(text, len(text))
            population = infer_population(section["section_title"], text)

            for part in split_by_sentences(text, max_tokens, overlap_tokens):
                chunk_counter = add_chunk(
                    chunks, chunk_counter, part, page["page"],
                    section, "general", population
                )
            continue

        if matches[0].start() > 0:
            pre = text[:matches[0].start()].strip()

            if pre:
                section = current_section_before(text, matches[0].start())
                population = infer_population(section["section_title"], pre)

                for part in split_by_sentences(pre, max_tokens, overlap_tokens):
                    chunk_counter = add_chunk(
                        chunks, chunk_counter, part, page["page"],
                        section, "general", population
                    )

        for i, match in enumerate(matches):
            start = match.start()
            end = matches[i + 1].start() if i + 1 < len(matches) else len(text)

            rec_id = match.group(1)
            rec_text = text[start:end].strip()

            section = current_section_before(text, start)
            population = infer_population(section["section_title"], rec_text)

            if token_count(rec_text) <= max_tokens:
                chunk_counter = add_chunk(
                    chunks, chunk_counter, rec_text, page["page"],
                    section, rec_id, population
                )
            else:
                for part in split_by_sentences(rec_text, max_tokens, overlap_tokens):
                    chunk_counter = add_chunk(
                        chunks, chunk_counter, part, page["page"],
                        section, rec_id, population
                    )

    return chunks

chunks = make_chunks(raw_pages)

chunk_df = pd.DataFrame(chunks)

print("Chunks:", len(chunks))
print("Mean tokens:", round(chunk_df["text"].map(token_count).mean(), 1))
print("Max tokens:", chunk_df["text"].map(token_count).max())

display(
    chunk_df.head(10)[[
        "chunk_id", "page", "section",
        "recommendation_id", "population", "text"
    ]]
)

Chunks: 175
Mean tokens: 117.2
Max tokens: 465


,chunk_id,page,section,recommendation_id,population,text
0,NICE_NG245_p1_0001,1,General guideline context,general,"Adults, young people and children (age-specifi...",Condition: Asthma. Section general: General gu...
1,NICE_NG245_p2_0002,2,General guideline context,general,"Adults, young people and children (age-specifi...",Condition: Asthma. Section general: General gu...
2,NICE_NG245_p3_0003,3,Organisation and delivery of care ...............,general,Children under 5,Condition: Asthma. Section 1.16: Organisation ...
3,NICE_NG245_p4_0004,4,General guideline context,general,Children under 5,Condition: Asthma. Section general: General gu...
4,NICE_NG245_p5_0005,5,General guideline context,general,Adults,Condition: Asthma. Section general: General gu...
5,NICE_NG245_p6_0006,6,General guideline context,general,"Adults, young people and children (age-specifi...",Condition: Asthma. Section general: General gu...
6,NICE_NG245_p7_0007,7,General guideline context,general,"Adults, young people and children (age-specifi...",Condition: Asthma. Section general: General gu...
7,NICE_NG245_p8_0008,8,Initial clinical assessment,general,"Adults, young people and children (age-specifi...",Condition: Asthma. Section 1.1: Initial clinic...
8,NICE_NG245_p8_0009,8,Initial clinical assessment,1.1.1,Adults,Condition: Asthma. Section 1.1: Initial clinic...
9,NICE_NG245_p8_0010,8,Obtain a structured clinical history in people...,1.1.2,"Adults, young people and children (age-specifi...",Condition: Asthma. Section 1.1.1: Obtain a str...


## 6. Biomedical embeddings — MedCPT

In [7]:
from transformers import AutoTokenizer, AutoModel

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

QUERY_MODEL_NAME = "ncbi/MedCPT-Query-Encoder"
ARTICLE_MODEL_NAME = "ncbi/MedCPT-Article-Encoder"

query_tokenizer = AutoTokenizer.from_pretrained(QUERY_MODEL_NAME)
query_model = AutoModel.from_pretrained(QUERY_MODEL_NAME).to(DEVICE)

article_tokenizer = AutoTokenizer.from_pretrained(ARTICLE_MODEL_NAME)
article_model = AutoModel.from_pretrained(ARTICLE_MODEL_NAME).to(DEVICE)

query_model.eval()
article_model.eval()

def mean_pool(output, attention_mask):
    token_embeddings = output.last_hidden_state
    mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return (token_embeddings * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)

@torch.no_grad()
def encode_texts(texts, tokenizer, model, batch_size=16):
    all_embeddings = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]

        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        ).to(DEVICE)

        outputs = model(**inputs)
        emb = mean_pool(outputs, inputs["attention_mask"])
        emb = torch.nn.functional.normalize(emb, p=2, dim=1)
        all_embeddings.append(emb.cpu().numpy())

    return np.vstack(all_embeddings)

def encode_query(query):
    return encode_texts(
        [query],
        query_tokenizer,
        query_model,
        batch_size=1
    )[0]

texts = [c["text"] for c in chunks]
embeddings = encode_texts(texts, article_tokenizer, article_model)

print("Embedding matrix:", embeddings.shape)

c:\Users\Mega Store\AppData\Local\Python\pythoncore-3.11-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3951.98it/s]


Embedding matrix: (175, 768)


## 7. FAISS + BM25 hybrid retrieval

In [8]:
dimension = embeddings.shape[1]

faiss_index = faiss.IndexFlatIP(dimension)
faiss_index.add(embeddings.astype("float32"))

bm25_corpus = [c["text"].lower().split() for c in chunks]
bm25 = BM25Okapi(bm25_corpus)

print("FAISS vectors:", faiss_index.ntotal)
print("BM25 documents:", len(bm25_corpus))

FAISS vectors: 175
BM25 documents: 175


In [9]:
def minmax(values):
    values = np.asarray(values, dtype=float)
    if len(values) == 0:
        return values

    lo, hi = values.min(), values.max()

    if hi - lo < 1e-9:
        return np.ones_like(values)

    return (values - lo) / (hi - lo)

def cosine_sim(a, b):
    return float(np.dot(a, b))

def mmr_select(query_embedding, candidate_indices, relevance_scores,
                k=FINAL_TOP_K, lambda_mult=MMR_LAMBDA):

    selected = []
    remaining = list(candidate_indices)

    while remaining and len(selected) < k:
        best_idx = None
        best_value = -1e9

        for idx in remaining:
            relevance = relevance_scores[idx]

            if not selected:
                diversity_penalty = 0.0
            else:
                diversity_penalty = max(
                    cosine_sim(embeddings[idx], embeddings[j])
                    for j in selected
                )

            value = (
                lambda_mult * relevance
                - (1 - lambda_mult) * diversity_penalty
            )

            if value > best_value:
                best_value = value
                best_idx = idx

        selected.append(best_idx)
        remaining.remove(best_idx)

    return selected

def hybrid_search(
    query,
    top_k=FINAL_TOP_K,
    condition=None,
    recommendation_id=None,
    population=None,
    min_score=MIN_RELEVANCE_SCORE,
    apply_threshold=True
):
    q = encode_query(query).astype("float32").reshape(1, -1)

    # Dense retrieval
    dense_scores, dense_ids = faiss_index.search(
        q, min(DENSE_TOP_K, len(chunks))
    )

    dense_scores = dense_scores[0]
    dense_ids = dense_ids[0]

    # Lexical retrieval
    bm_scores_all = bm25.get_scores(query.lower().split())
    bm_ids = np.argsort(bm_scores_all)[::-1][:min(BM25_TOP_K, len(chunks))]

    candidate_ids = set(int(i) for i in dense_ids if i >= 0)
    candidate_ids.update(int(i) for i in bm_ids)

    # Metadata filtering
    filtered = []

    for idx in candidate_ids:
        item = chunks[idx]

        if recommendation_id is not None:
            if item["recommendation_id"] != recommendation_id:
                continue

        if condition is not None:
            if condition.lower() not in item["condition"].lower():
                continue

        if population is not None:
            if population.lower() not in item["population"].lower():
                continue

        filtered.append(idx)

    if not filtered:
        return []

    dense_map = {
        int(i): float(s)
        for i, s in zip(dense_ids, dense_scores)
    }

    bm_map = {
        int(i): float(bm_scores_all[i])
        for i in bm_ids
    }

    dense_raw = np.array([dense_map.get(i, 0.0) for i in filtered])
    bm_raw = np.array([bm_map.get(i, 0.0) for i in filtered])

    dense_norm = minmax(dense_raw)
    bm_norm = minmax(bm_raw)

    fused = (
        DENSE_WEIGHT * dense_norm
        + BM25_WEIGHT * bm_norm
    )

    score_map = {
        idx: float(score)
        for idx, score in zip(filtered, fused)
    }

    # Relevance threshold BEFORE MMR.
    # This is the main abstention fix: unrelated questions should not
    # automatically receive the nearest asthma chunks.
    if apply_threshold:
        filtered = [
            idx for idx in filtered
            if score_map[idx] >= min_score
        ]

    if not filtered:
        return []

    selected = mmr_select(
        q[0],
        filtered,
        score_map,
        k=top_k
    )

    results = []

    for idx in selected:
        item = dict(chunks[idx])
        item["dense_score"] = dense_map.get(idx, 0.0)
        item["bm25_score"] = bm_map.get(idx, 0.0)
        item["hybrid_score"] = score_map[idx]
        results.append(item)

    return results

def show_results(results):
    if not results:
        print("No sufficiently relevant evidence retrieved.")
        return

    rows = []

    for r in results:
        rows.append({
            "page": r["page"],
            "section": r["section_id"],
            "recommendation": r["recommendation_id"],
            "hybrid_score": round(r["hybrid_score"], 4),
            "dense_score": round(r["dense_score"], 4),
            "bm25_score": round(r["bm25_score"], 4),
            "text": r["source_text"][:500]
        })

    display(pd.DataFrame(rows))

## 8. Scope, safety and abstention gate

In [10]:
RED_FLAG_PATTERNS = [
    "acute asthma attack",
    "severe asthma attack",
    "difficulty breathing",
    "struggling to breathe",
    "cannot speak",
    "hospital admission",
    "emergency department",
    "acute exacerbation"
]

# Strong asthma/domain terms.
IN_SCOPE_TERMS = [
    "asthma", "wheeze", "wheezing", "breathlessness",
    "chest tightness", "fev1", "feno", "eosinophil",
    "peak expiratory flow", "pef", "spirometry",
    "bronchodilator", "bronchodilator reversibility",
    "inhaled corticosteroid", "ics", "formoterol", "mart",
    "air therapy", "saba", "laba", "lama", "ltra",
    "inhaler", "inhaler technique", "asthma control",
    "asthma action plan", "airway", "lung function",
    "bronchial challenge", "fractional exhaled nitric oxide"
]

KNOWN_OUT_OF_SCOPE_TERMS = [
    "migraine", "headache", "diabetes", "hypertension",
    "pneumonia", "urinary tract infection", "uti",
    "heart attack", "stroke", "kidney disease",
    "liver disease", "pregnancy ultrasound"
]

def safety_check(query):
    q = query.lower().strip()

    red_flags = [p for p in RED_FLAG_PATTERNS if p in q]
    out_of_scope = [p for p in KNOWN_OUT_OF_SCOPE_TERMS if p in q]
    scope_terms_found = [p for p in IN_SCOPE_TERMS if p in q]

    # Hard reject obvious unrelated clinical topics.
    if out_of_scope:
        in_scope = False
    else:
        # If an asthma/domain term is present, allow retrieval.
        # If no term is present, retrieval can still be attempted because
        # users may ask paraphrased questions; the relevance threshold is
        # the second safety layer.
        in_scope = True

    return {
        "in_scope": in_scope,
        "scope_terms_found": scope_terms_found,
        "out_of_scope_terms": out_of_scope,
        "red_flags_detected": red_flags,
        "scope": f"{CLINICAL_FIELD} → {CONDITION}",
        "population": POPULATION
    }

def build_context(results):
    blocks = []

    for i, r in enumerate(results, start=1):
        citation = (
            f"[NICE NG245 | Section {r['section_id']} | "
            f"Recommendation {r['recommendation_id']} | "
            f"Page {r['page']} | Population: {r['population']}]"
        )

        blocks.append(
            f"Evidence {i} {citation}\n{r['source_text']}"
        )

    return "\n\n---\n\n".join(blocks)

def citation_for(r):
    return (
        f"[NICE NG245, Section {r['section_id']}, "
        f"Recommendation {r['recommendation_id']}, p.{r['page']}]"
    )

## 9. Retrieval test — inspect evidence before generation

In [11]:
query = "What objective tests are recommended to diagnose asthma in adults?"

safety = safety_check(query)
print("Safety:", safety)

results = hybrid_search(
    query,
    top_k=FINAL_TOP_K,
    condition=CONDITION
)

show_results(results)

Safety: {'in_scope': True, 'scope_terms_found': ['asthma'], 'out_of_scope_terms': [], 'red_flags_detected': [], 'scope': 'Allergic Diseases → Asthma', 'population': 'Adults, young people and children (age-specific recommendations apply)'}


,page,section,recommendation,hybrid_score,dense_score,bm25_score,text
0,13,1.3,1.3.1,0.9545,0.7914,11.4640,1.3.1 \nFor children under 5 with suspected as...
1,46,general,general,0.9538,0.7937,11.3663,In view of the difficulty in diagnosing asthma...
2,13,1.3.2,1.3.3,0.9105,0.7783,10.2296,1.3.3 \nRefer to a specialist respiratory paed...
3,41,general,general,0.9136,0.7907,9.9688,Rationale and impact \nThese sections briefly ...
4,41,general,1.2.9,0.8898,0.7866,9.2104,1.2.9 \nWhy the committee made the recommendat...


## 10. Optional LLM generation with a validation gate

In [12]:
from getpass import getpass

if USE_LLM:
    from openai import OpenAI

    OPENAI_API_KEY = getpass("Enter OPENAI_API_KEY: ")
    client = OpenAI(api_key=OPENAI_API_KEY)

SYSTEM_PROMPT = '''
You are a clinical evidence retrieval assistant for an asthma guideline.

Use ONLY the supplied retrieved evidence.
Do not use outside knowledge.
Do not invent facts, recommendations, dosages, contraindications, age groups,
or citations.
Do not diagnose a patient.
Do not prescribe individualized treatment.
Preserve all numerical values and negative recommendations exactly.

Every clinically relevant claim must include a citation:
[NICE NG245, Section X.X, Recommendation X.X.X, p.X]

If the evidence is insufficient, say:
"I don't have enough information in the available source to answer this question reliably."

Confidence must be High, Medium, or Low.

Return exactly:

Answer: <answer>

Confidence: <High / Medium / Low>

Reason: <brief explanation>
'''

def extract_numbers(text):
    return re.findall(
        r"(?<!\w)(?:\d+(?:\.\d+)?)(?:\s*[-–]\s*\d+(?:\.\d+)?)?",
        text
    )

def validate_citations(answer, sources):
    valid = {citation_for(r) for r in sources}

    found = re.findall(
        r"\[NICE NG245, Section [^\]]+, Recommendation [^\]]+, p\.\d+\]",
        answer
    )

    invalid = [c for c in found if c not in valid]

    # If the answer contains clinical prose but no citation, fail validation.
    missing_citation = len(found) == 0 and len(answer.strip()) > 0

    return {
        "citations_found": found,
        "invalid_citations": invalid,
        "missing_citation": missing_citation,
        "citation_valid": len(invalid) == 0 and not missing_citation
    }

def validate_numbers(answer, sources):
    source_text = " ".join(r["source_text"] for r in sources)

    answer_numbers = set(extract_numbers(answer))
    source_numbers = set(extract_numbers(source_text))

    unsupported = sorted(answer_numbers - source_numbers)

    return {
        "answer_numbers": sorted(answer_numbers),
        "unsupported_numbers": unsupported,
        "numeric_check": len(unsupported) == 0
    }

def validate_response(answer, sources):
    citation_result = validate_citations(answer, sources)
    number_result = validate_numbers(answer, sources)

    return {
        **citation_result,
        **number_result,
        "safe_to_display": (
            citation_result["citation_valid"]
            and number_result["numeric_check"]
        )
    }

def answer_query(query, use_llm=USE_LLM):
    safety = safety_check(query)

    if not safety["in_scope"]:
        return {
            "answer": (
                "ABSTAIN: The current knowledge base is scoped to asthma "
                "guidance. The available source does not provide enough "
                "information to answer this question reliably."
            ),
            "sources": [],
            "confidence": "Low",
            "status": "abstained"
        }

    results = hybrid_search(
        query,
        top_k=FINAL_TOP_K,
        condition=CONDITION,
        min_score=MIN_RELEVANCE_SCORE,
        apply_threshold=True
    )

    # Second safety layer: no relevant evidence = abstain.
    if not results:
        return {
            "answer": (
                "ABSTAIN: I don't have enough information in the available "
                "NICE NG245 evidence to answer this question reliably."
            ),
            "sources": [],
            "confidence": "Low",
            "status": "abstained"
        }

    context = build_context(results)

    # Retrieval-only mode
    if not use_llm:
        return {
            "answer": (
                "LLM generation is disabled. Retrieved evidence is shown "
                "below for inspection.\n\n" + context
            ),
            "sources": results,
            "confidence": "Evidence retrieval only",
            "status": "retrieved"
        }

    user_prompt = f'''
Clinical evidence:
{context}

User question:
{query}

Answer only from the evidence above.
Include precise citations for every clinical claim.
'''

    response = client.chat.completions.create(
        model=OPENAI_MODEL,
        temperature=0,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt}
        ]
    )

    answer = response.choices[0].message.content.strip()

    validation = validate_response(answer, results)

    # HARD DISPLAY GATE:
    # If citation or numerical validation fails, do not display the unsafe
    # generated answer as the final answer.
    if not validation["safe_to_display"]:
        return {
            "answer": (
                "ABSTAIN: The generated answer failed evidence validation "
                "and was withheld."
            ),
            "sources": results,
            "confidence": "Low",
            "status": "validation_failed",
            "validation": validation
        }

    return {
        "answer": answer,
        "sources": results,
        "confidence": "validated",
        "status": "validated",
        "validation": validation
    }

## 11. Evidence panel

In [13]:
def evidence_panel(query, results):
    print("=" * 90)
    print("EVIDENCE PANEL")
    print("=" * 90)
    print("Clinical query:", query)
    print()

    if not results:
        print("No sufficiently relevant evidence retrieved.")
        return

    for i, r in enumerate(results, start=1):
        print(f"Chunk {i}")
        print(f"Score: {r['hybrid_score']:.4f}")
        print(f"Document: {r['title']}")
        print(f"Section: {r['section_id']} — {r['section']}")
        print(f"Recommendation: {r['recommendation_id']}")
        print(f"Page: {r['page']}")
        print(f"Population: {r['population']}")
        print(f"Citation: {citation_for(r)}")
        print("Evidence:")
        print(r["source_text"][:900])
        print("-" * 90)

## 12. Labeled evaluation set — 20 questions

In [14]:
# Expected recommendation IDs are used for manual/benchmark labeling.
# These labels should be checked against the exact guideline page if your
# team changes the source PDF/version.

EVAL_SET = [
    {
        "id": 1,
        "question": "What clinical history should be obtained when asthma is suspected?",
        "relevant_recommendations": ["1.1.1", "1.1.2"]
    },
    {
        "id": 2,
        "question": "What objective tests can diagnose asthma in adults?",
        "relevant_recommendations": ["1.2.1", "1.2.2", "1.2.3", "1.2.4"]
    },
    {
        "id": 3,
        "question": "What FeNO level supports a diagnosis of asthma in adults?",
        "relevant_recommendations": ["1.2.1"]
    },
    {
        "id": 4,
        "question": "What FEV1 increase is required for bronchodilator reversibility in adults?",
        "relevant_recommendations": ["1.2.2"]
    },
    {
        "id": 5,
        "question": "How should asthma control be monitored at every review?",
        "relevant_recommendations": ["1.5.1", "1.5.2", "1.5.3", "1.5.4"]
    },
    {
        "id": 6,
        "question": "Which validated symptom questionnaires can be considered during an asthma review?",
        "relevant_recommendations": ["1.5.5"]
    },
    {
        "id": 7,
        "question": "Should regular peak expiratory flow monitoring be used routinely to assess asthma control?",
        "relevant_recommendations": ["1.5.6"]
    },
    {
        "id": 8,
        "question": "What should be checked before increasing asthma treatment when control is suboptimal?",
        "relevant_recommendations": ["1.6.1", "1.6.2"]
    },
    {
        "id": 9,
        "question": "What should be checked at every asthma-related healthcare review regarding adherence and inhaler use?",
        "relevant_recommendations": ["1.6.1"]
    },
    {
        "id": 10,
        "question": "What is the initial treatment for newly diagnosed asthma in people aged 12 and over?",
        "relevant_recommendations": ["1.7.1", "1.7.2"]
    },
    {
        "id": 11,
        "question": "What treatment is recommended for people aged 12 and over whose asthma is not controlled on low-dose MART?",
        "relevant_recommendations": ["1.7.5", "1.7.6"]
    },
    {
        "id": 12,
        "question": "How long should a trial of an LTRA or LAMA be given before assessing response?",
        "relevant_recommendations": ["1.7.7"]
    },
    {
        "id": 13,
        "question": "What should an asthma self-management programme include?",
        "relevant_recommendations": ["1.10.1", "1.10.2", "1.10.3"]
    },
    {
        "id": 14,
        "question": "How long should be allowed before considering a further reduction in maintenance asthma treatment?",
        "relevant_recommendations": ["1.9.1", "1.9.2"]
    },
    {
        "id": 15,
        "question": "What should be considered regarding asthma treatment during pregnancy?",
        "relevant_recommendations": ["1.8.1"]
    },
    {
        "id": 16,
        "question": "What FeNO level supports a diagnosis of asthma in children aged 5 to 16?",
        "relevant_recommendations": ["1.3.1"]
    },
    {
        "id": 17,
        "question": "What spirometry result supports a diagnosis of asthma in children aged 5 to 16 when FeNO is not raised or unavailable?",
        "relevant_recommendations": ["1.3.2"]
    },
    {
        "id": 18,
        "question": "What should be done if a child under 5 with suspected asthma is unable to perform objective tests when they reach age 5?",
        "relevant_recommendations": ["1.4.1"]
    },
    {
        "id": 19,
        "question": "What factors should be checked when monitoring asthma control at every review?",
        "relevant_recommendations": ["1.5.1", "1.5.2", "1.5.3", "1.5.4"]
    },
    {
        "id": 20,
        "question": "Which people with asthma are at increased risk of poor outcomes?",
        "relevant_recommendations": ["1.11.1", "1.11.2"]
    }
]

## 13. Precision@K evaluation

In [15]:
def precision_at_k(results, relevant_ids, k):
    top_results = results[:k]

    if k == 0:
        return 0.0

    relevant = sum(
        1 for r in top_results
        if r["recommendation_id"] in set(relevant_ids)
    )

    return relevant / k

def evaluate_retrieval(eval_set, threshold=MIN_RELEVANCE_SCORE):
    rows = []

    for item in eval_set:
        # For evaluation we retrieve without the threshold first so that
        # Precision@K measures the ranking itself.
        results = hybrid_search(
            item["question"],
            top_k=10,
            condition=CONDITION,
            min_score=threshold,
            apply_threshold=False
        )

        p3 = precision_at_k(
            results,
            item["relevant_recommendations"],
            3
        )

        p5 = precision_at_k(
            results,
            item["relevant_recommendations"],
            5
        )

        rows.append({
            "id": item["id"],
            "question": item["question"],
            "Precision@3": p3,
            "Precision@5": p5,
            "Top-3 recommendations": [
                r["recommendation_id"] for r in results[:3]
            ],
            "Top-5 recommendations": [
                r["recommendation_id"] for r in results[:5]
            ],
            "top_pages": [r["page"] for r in results[:5]]
        })

    return pd.DataFrame(rows)

eval_df = evaluate_retrieval(EVAL_SET)

display(eval_df)

print(
    "Mean Precision@3:",
    round(eval_df["Precision@3"].mean(), 3)
)

print(
    "Mean Precision@5:",
    round(eval_df["Precision@5"].mean(), 3)
)

,id,question,Precision@3,Precision@5,Top-3 recommendations,Top-5 recommendations,top_pages
0,1,What clinical history should be obtained when ...,0.333333,0.2,"[1.1.2, 1.2.9, 1.6.3]","[1.1.2, 1.2.9, 1.6.3, general, 1.3.2]","[8, 41, 49, 8, 13]"
1,2,What objective tests can diagnose asthma in ad...,0.333333,0.2,"[1.2.4, 1.3.1, 1.1.7]","[1.2.4, 1.3.1, 1.1.7, general, 1.2.9]","[10, 13, 9, 41, 41]"
2,3,What FeNO level supports a diagnosis of asthma...,0.000000,0.2,"[general, 1.2.5, general]","[general, 1.2.5, general, 1.2.6, 1.2.1]","[48, 10, 46, 10, 10]"
3,4,What FEV1 increase is required for bronchodila...,0.333333,0.2,"[1.2.2, general, 1.2.6]","[1.2.2, general, 1.2.6, general, general]","[10, 42, 10, 32, 46]"
4,5,How should asthma control be monitored at ever...,0.666667,0.4,"[1.5.2, 1.5.1, 1.6.7]","[1.5.2, 1.5.1, 1.6.7, 1.6.1, 1.10.3]","[14, 14, 17, 14, 26]"
5,6,Which validated symptom questionnaires can be ...,0.000000,0.0,"[1.5.2, 1.5.3, general]","[1.5.2, 1.5.3, general, 1.1.4, general]","[14, 14, 47, 9, 15]"
6,7,Should regular peak expiratory flow monitoring...,0.000000,0.0,"[1.5.3, 1.5.4, 1.1.5]","[1.5.3, 1.5.4, 1.1.5, general, 1.2.4]","[14, 14, 9, 32, 10]"
7,8,What should be checked before increasing asthm...,0.000000,0.0,"[1.14.4, 1.6.3, 1.14.5]","[1.14.4, 1.6.3, 1.14.5, 1.7.11, general]","[29, 49, 58, 54, 59]"
8,9,What should be checked at every asthma-related...,0.000000,0.0,"[1.6.7, general, 1.11.1]","[1.6.7, general, 1.11.1, 1.16.1, 1.5.1]","[17, 39, 26, 31, 14]"
9,10,What is the initial treatment for newly diagno...,0.000000,0.0,"[general, general, general]","[general, general, general, general, general]","[19, 18, 50, 51, 58]"


Mean Precision@3: 0.117
Mean Precision@5: 0.08


## 14. Compare Top-K settings 

In [16]:
def evaluate_top_k(eval_set, k_values=(3, 5, 10)):
    rows = []

    for k in k_values:
        p_values = []

        for item in eval_set:
            results = hybrid_search(
                item["question"],
                top_k=k,
                condition=CONDITION,
                apply_threshold=False
            )

            p = precision_at_k(
                results,
                item["relevant_recommendations"],
                k
            )

            p_values.append(p)

        rows.append({
            "Top-K": k,
            "Mean Precision@K": np.mean(p_values)
        })

    return pd.DataFrame(rows)

topk_results = evaluate_top_k(EVAL_SET)

display(topk_results)

,Top-K,Mean Precision@K
0,3,0.116667
1,5,0.080000
2,10,0.045000


## 15. Compare chunk configurations 

In [17]:
# This comparison rebuilds chunks and embeddings for each configuration.
# It is intentionally optional because embedding the guideline twice takes time.

CHUNK_EXPERIMENTS = {
    "A_450_no_overlap": {"max_tokens": 450, "overlap": 0},
    "B_700_70_overlap": {"max_tokens": 700, "overlap": 70},
}

def build_retrieval_for_chunks(test_chunks):
    test_embeddings = encode_texts(
        [c["text"] for c in test_chunks],
        article_tokenizer,
        article_model
    )

    test_faiss = faiss.IndexFlatIP(test_embeddings.shape[1])
    test_faiss.add(test_embeddings.astype("float32"))

    test_bm25 = BM25Okapi(
        [c["text"].lower().split() for c in test_chunks]
    )

    return test_chunks, test_embeddings, test_faiss, test_bm25

def simple_eval_with_indexes(eval_set, test_chunks, test_embeddings, test_faiss, test_bm25):
    rows = []

    for item in eval_set:
        q = encode_query(item["question"]).astype("float32").reshape(1, -1)

        dense_scores, dense_ids = test_faiss.search(
            q, min(30, len(test_chunks))
        )

        dense_scores = dense_scores[0]
        dense_ids = dense_ids[0]

        bm_scores = test_bm25.get_scores(item["question"].lower().split())
        bm_ids = np.argsort(bm_scores)[::-1][:min(30, len(test_chunks))]

        candidate_ids = set(int(i) for i in dense_ids if i >= 0)
        candidate_ids.update(int(i) for i in bm_ids)

        dense_map = {int(i): float(s) for i, s in zip(dense_ids, dense_scores)}
        bm_map = {int(i): float(bm_scores[i]) for i in bm_ids}

        filtered = list(candidate_ids)

        dense_raw = np.array([dense_map.get(i, 0.0) for i in filtered])
        bm_raw = np.array([bm_map.get(i, 0.0) for i in filtered])

        fused = (
            DENSE_WEIGHT * minmax(dense_raw)
            + BM25_WEIGHT * minmax(bm_raw)
        )

        ranking = [
            idx for _, idx in sorted(
                zip(fused, filtered),
                reverse=True
            )
        ]

        top5 = [test_chunks[i] for i in ranking[:5]]

        p5 = precision_at_k(
            top5,
            item["relevant_recommendations"],
            5
        )

        rows.append(p5)

    return float(np.mean(rows))

print("Chunk comparison is optional and may take several minutes.")
print("Run this cell if you want the Day 2 chunk-size comparison table.")

# Uncomment to run:
#
# chunk_comparison_rows = []
#
# for name, cfg in CHUNK_EXPERIMENTS.items():
#     test_chunks = make_chunks(
#         raw_pages,
#         max_tokens=cfg["max_tokens"],
#         overlap_tokens=cfg["overlap"]
#     )
#     tc, te, ti, tb = build_retrieval_for_chunks(test_chunks)
#     mean_p5 = simple_eval_with_indexes(EVAL_SET, tc, te, ti, tb)
#
#     chunk_comparison_rows.append({
#         "Configuration": name,
#         "Max tokens": cfg["max_tokens"],
#         "Overlap": cfg["overlap"],
#         "Chunks": len(test_chunks),
#         "Mean Precision@5": mean_p5
#     })
#
# chunk_comparison_df = pd.DataFrame(chunk_comparison_rows)
# display(chunk_comparison_df)

Chunk comparison is optional and may take several minutes.
Run this cell if you want the Day 2 chunk-size comparison table.


## 16. Calibrate the relevance threshold

In [18]:
def threshold_sweep(eval_set, thresholds=(0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50)):
    rows = []

    for threshold in thresholds:
        answered = 0
        p5_values = []

        for item in eval_set:
            results = hybrid_search(
                item["question"],
                top_k=5,
                condition=CONDITION,
                min_score=threshold,
                apply_threshold=True
            )

            if results:
                answered += 1

            p5_values.append(
                precision_at_k(
                    results,
                    item["relevant_recommendations"],
                    5
                )
            )

        rows.append({
            "Threshold": threshold,
            "Coverage": answered / len(eval_set),
            "Mean Precision@5": np.mean(p5_values)
        })

    return pd.DataFrame(rows)

threshold_df = threshold_sweep(EVAL_SET)
display(threshold_df)

print("Use the table to choose a threshold that balances retrieval quality and coverage.")

,Threshold,Coverage,Mean Precision@5
0,0.20,1.0,0.08
1,0.25,1.0,0.08
2,0.30,1.0,0.08
3,0.35,1.0,0.08
4,0.40,1.0,0.08
5,0.45,1.0,0.08
6,0.50,1.0,0.08


Use the table to choose a threshold that balances retrieval quality and coverage.


## 17. Out-of-scope and failure-case tests

In [19]:
OUT_OF_SCOPE_QUESTIONS = [
    "What are the symptoms of migraine?",
    "What is the treatment for hypertension?",
    "What are the symptoms of pneumonia?",
    "How is diabetes diagnosed?",
    "What antibiotics are used for urinary tract infection?",
    "What are the red flags that should prompt further investigation for headache?"
]

for q in OUT_OF_SCOPE_QUESTIONS:
    print("\n" + "=" * 90)
    print("OUT-OF-SCOPE TEST:", q)

    result = answer_query(q, use_llm=False)

    print(result["answer"])
    print("Status:", result["status"])


OUT-OF-SCOPE TEST: What are the symptoms of migraine?
ABSTAIN: The current knowledge base is scoped to asthma guidance. The available source does not provide enough information to answer this question reliably.
Status: abstained

OUT-OF-SCOPE TEST: What is the treatment for hypertension?
ABSTAIN: The current knowledge base is scoped to asthma guidance. The available source does not provide enough information to answer this question reliably.
Status: abstained

OUT-OF-SCOPE TEST: What are the symptoms of pneumonia?
ABSTAIN: The current knowledge base is scoped to asthma guidance. The available source does not provide enough information to answer this question reliably.
Status: abstained

OUT-OF-SCOPE TEST: How is diabetes diagnosed?
ABSTAIN: The current knowledge base is scoped to asthma guidance. The available source does not provide enough information to answer this question reliably.
Status: abstained

OUT-OF-SCOPE TEST: What antibiotics are used for urinary tract infection?
ABSTAIN

## 18. Final interactive demo

In [20]:
user_query = "What objective tests are recommended to diagnose asthma in adults?"

demo = answer_query(
    user_query,
    use_llm=USE_LLM
)

print("=" * 90)
print("FINAL ANSWER")
print("=" * 90)
print(demo["answer"])
print("\nStatus:", demo["status"])
print("Confidence:", demo["confidence"])

if demo["sources"]:
    print("\n")
    evidence_panel(user_query, demo["sources"])

if "validation" in demo:
    print("\nValidation:")
    print(json.dumps(demo["validation"], indent=2))

FINAL ANSWER
LLM generation is disabled. Retrieved evidence is shown below for inspection.

Evidence 1 [NICE NG245 | Section 1.3 | Recommendation 1.3.1 | Page 13 | Population: Children under 5]
1.3.1 
For children under 5 with suspected asthma, treat with inhaled corticosteroids in 
line with the recommendations on medicines for initial management in children 
under 5, and review the child on a regular basis. If they still have symptoms when 
they reach 5 years, attempt objective tests (see the section on objective tests for 
diagnosing asthma in adults, young people and children aged 5 to 16). [NICE 
2017]

---

Evidence 2 [NICE NG245 | Section general | Recommendation general | Page 46 | Population: Children under 5]
In view of the difficulty in diagnosing asthma in this age group the committee also agreed 
that any child who had been admitted to hospital, or been taken to the emergency 
department twice or more, because of wheezing or breathlessness should be referred to a 
speciali

## 19. Batch demo — 20 questions

In [21]:
for item in EVAL_SET:
    q = item["question"]

    result = answer_query(
        q,
        use_llm=False
    )

    print("\n" + "=" * 90)
    print(f"QUESTION {item['id']}: {q}")
    print("STATUS:", result["status"])

    if result["sources"]:
        print(
            "Top evidence:",
            result["sources"][0]["recommendation_id"],
            "| score:",
            round(result["sources"][0]["hybrid_score"], 4),
            "| page:",
            result["sources"][0]["page"]
        )
    else:
        print(result["answer"])


QUESTION 1: What clinical history should be obtained when asthma is suspected?
STATUS: retrieved
Top evidence: 1.1.2 | score: 0.9904 | page: 8

QUESTION 2: What objective tests can diagnose asthma in adults?
STATUS: retrieved
Top evidence: 1.2.4 | score: 0.955 | page: 10

QUESTION 3: What FeNO level supports a diagnosis of asthma in adults?
STATUS: retrieved
Top evidence: general | score: 0.9837 | page: 48

QUESTION 4: What FEV1 increase is required for bronchodilator reversibility in adults?
STATUS: retrieved
Top evidence: 1.2.2 | score: 1.0 | page: 10

QUESTION 5: How should asthma control be monitored at every review?
STATUS: retrieved
Top evidence: 1.5.2 | score: 0.9922 | page: 14

QUESTION 6: Which validated symptom questionnaires can be considered during an asthma review?
STATUS: retrieved
Top evidence: 1.5.2 | score: 1.0 | page: 14

QUESTION 7: Should regular peak expiratory flow monitoring be used routinely to assess asthma control?
STATUS: abstained
ABSTAIN: The current knowl

## 20. Save RAG artifacts

In [22]:
faiss.write_index(faiss_index, "medical_rag.faiss")

with open("medical_rag_chunks.pkl", "wb") as f:
    pickle.dump(chunks, f)

np.save("medical_rag_embeddings.npy", embeddings)

with open("rag_config.json", "w", encoding="utf-8") as f:
    json.dump({
        "document_id": DOCUMENT_ID,
        "document_title": DOCUMENT_TITLE,
        "source_url": SOURCE_URL,
        "condition": CONDITION,
        "max_chunk_tokens": MAX_CHUNK_TOKENS,
        "chunk_overlap_tokens": CHUNK_OVERLAP_TOKENS,
        "dense_top_k": DENSE_TOP_K,
        "bm25_top_k": BM25_TOP_K,
        "final_top_k": FINAL_TOP_K,
        "mmr_lambda": MMR_LAMBDA,
        "min_relevance_score": MIN_RELEVANCE_SCORE,
        "dense_weight": DENSE_WEIGHT,
        "bm25_weight": BM25_WEIGHT
    }, f, indent=2)

eval_df.to_csv("retrieval_evaluation.csv", index=False)

print("Saved:")
print("- medical_rag.faiss")
print("- medical_rag_chunks.pkl")
print("- medical_rag_embeddings.npy")
print("- rag_config.json")
print("- retrieval_evaluation.csv")

Saved:
- medical_rag.faiss
- medical_rag_chunks.pkl
- medical_rag_embeddings.npy
- rag_config.json
- retrieval_evaluation.csv
